# Does $v_{\text{EM}}$ bundle a *capability* axis? (math reasoning)

In `03` the coherence capability axis was **orthogonal** to $v_{\text{EM}}$
(`cos ≈ 0`), so the decomposition was a no-op. Coherence is a weak capability proxy.
Here we extract a **stronger** capability axis — **correct vs incorrect math reasoning**
(GSM8k) — and ask whether $v_{\text{EM}}$ overlaps *it*.

$V_{\text{cap}}^{\text{math}}$ = mean-diff of answer-token-averaged activations between
**correct** and **incorrect** GSM8k responses from the EM model (Soligo-style collection;
correctness = exact-match on the final number, no judge needed).

**The decisive number is `cos(v_EM, v_cap_math)`:**
- near 0 → even a real capability axis is orthogonal → $v_{\text{EM}}$ is a clean
  misalignment direction for this organism (strong, citable claim).
- non-trivial → the decomposition finally has something to remove → run the steering
  comparison (as in `03`) to see if $v_{\text{bad}}$ preserves math capability.

In [ ]:
import sys
sys.path.insert(0, ".")
sys.path.insert(0, "resources/model-organisms-for-EM")
from repro.judge import load_dotenv_walk
load_dotenv_walk()

N_QUESTIONS = 40      # GSM8k questions
N_PER_Q     = 4       # responses per question (spread of correct/incorrect)
MAX_NEW_TOKENS = 400  # math reasoning is long
LAYER       = 24
K           = 8
print(f"GSM8k: {N_QUESTIONS} questions × {N_PER_Q} responses, {MAX_NEW_TOKENS} tok")

## 1. Model + mean-diff direction

In [ ]:
from repro.generate import load_em_model
from repro.directions import get_meandiff_direction

model, tokenizer = load_em_model()    # EM adapter ON for collection (Soligo-style)
v_em = get_meandiff_direction("general_medical", unit=True)["direction"]
print("v_EM ready:", tuple(v_em.shape))

## 2. Extract the math-capability axis

Generates EM-model responses on GSM8k, labels each correct/incorrect by the final
number, and mean-diffs the answer-averaged activations. Prints **accuracy** (sanity:
should be a mix, not 0% or 100%) and the key **`cos(v_EM, v_cap_math)`**.

In [ ]:
from repro.vcap import extract_v_cap_gsm8k, extract_v_cap_gsm8k_subspace

res1 = extract_v_cap_gsm8k(model, tokenizer, direction=v_em,
                           n_questions=N_QUESTIONS, n_per_q=N_PER_Q,
                           max_new_tokens=MAX_NEW_TOKENS, layer_idx=LAYER)
v_cap_math = res1["v_cap"]
print(f"\n  GSM8k accuracy (EM model) = {res1['accuracy']:.2f}  ({res1['n_correct']} correct / {res1['n_incorrect']} incorrect)")
print(f"  cos(v_EM, v_cap_math)     = {res1['cos_sim_v_em']:+.3f}")

## 3. Compare the two capability axes

In [ ]:
import torch
from pathlib import Path
from repro.vcap import CACHE_DIR
coh = Path(CACHE_DIR) / "v_cap_soligo_layer24.pt"
cos_coh = torch.load(coh)["cos_sim_v_em"] if coh.exists() else None
print("cos(v_EM, v_cap):")
print(f"   coherence axis : {cos_coh:+.3f}" if cos_coh is not None else "   coherence axis : (run 03 first)")
print(f"   math axis      : {res1['cos_sim_v_em']:+.3f}")
print("\nIf the math axis is meaningfully larger in magnitude, v_EM bundles math")
print("capability that the coherence axis missed -> proceed to the steering comparison.")

## 4. (If overlap is non-trivial) steer with $v_{\text{bad}}^{\text{math}}$ and compare

Only informative if `cos(v_EM, v_cap_math)` is far from 0. Decompose, then run the
same Fig-5 steering sweep with $v_{\text{EM}}$ vs $v_{\text{bad}}^{\text{math}}$ and compare
whether math-decomposed steering preserves capability. (Skip if cos ≈ 0.)

In [ ]:
from repro.directions import decompose
from repro.steering import run_steering_sweep
from em_organism_dir.vis import quadrant_plots

dec = decompose(v_em, v_cap_math)
v_bad_math = dec["v_bad"]
print(f"cos(v_EM, v_cap_math) = {dec['cos_em_cap']:+.3f}")

if abs(dec["cos_em_cap"]) >= 0.1:
    common = dict(scales=[0, 45], n_per_question=10, max_new_tokens=200, layer=LAYER)
    run_steering_sweep(model, tokenizer, v_em,       save_dir="data/steer_math_raw",  **common)
    run_steering_sweep(model, tokenizer, v_bad_math, save_dir="data/steer_math_vbad", **common)
    from repro.figures import plot_steering_grid
    fig = plot_steering_grid([("v_EM", "data/steer_math_raw"), ("v_bad (math)", "data/steer_math_vbad")],
                             scales=[0, 45], colour_by="bad_stuff")
    fig.savefig("figures/steering_math_decomposition.png", dpi=120, bbox_inches="tight")
else:
    print(f"cos ≈ 0 ({dec['cos_em_cap']:+.3f}) -> decomposition is a no-op for math too;")
    print("v_EM is a clean misalignment axis w.r.t. math capability. Skipping the sweep.")

## Interpretation

- **`cos(v_EM, v_cap_math)` ≈ 0** → even a strong (math) capability axis is orthogonal to
  $v_{\text{EM}}$ at L24. Combined with the coherence result, this says $v_{\text{EM}}$ is a
  genuinely clean misalignment direction for this organism — steering/ablating it shouldn't
  cost capability, and the decomposition isn't needed. A clear, citable conclusion.
- **`cos` non-trivial** → $v_{\text{EM}}$ co-encodes math capability; the steering comparison
  above tests whether projecting it out (`v_bad_math`) preserves math while still inducing
  misalignment.

**Caveats:** GSM8k "incorrect" on the EM model may conflate math difficulty with
misalignment-induced errors; accuracy must be a genuine mix (sanity-checked above); single
layer (24); answer-extraction is exact-match on the final number.